# IQ Demodulation in an Rx Chain — Carrier, Mixer, Residual $\Delta f$ & FFT Bins

A receiver brings a real bandpass pulse centred at carrier $f_c$ down to **complex baseband** by multiplying it with two quadrature copies of a local oscillator at $f_{LO}$, then low-pass filtering:

$$z(t) = I(t) + jQ(t) = \mathrm{LPF}\!\big\{x(t)\,e^{-j2\pi f_{LO} t}\big\},\qquad I = x\cos(2\pi f_{LO}t),\;\; Q = -x\sin(2\pi f_{LO}t)$$

If the oscillator is not perfectly tuned, a **residual** $\Delta f = f_c - f_{LO}$ survives demodulation and rotates the baseband phasor at $\Delta f$ — the I/Q channels oscillate instead of staying flat. The sliders below let you push the pulse along the frequency axis through every stage (RF $\to$ IF $\to$ baseband) and watch where its energy lands relative to the **DFT bin grid** of width $f_s/N$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.lines import Line2D
%matplotlib inline

plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'figure.dpi': 100, 'font.size': 11,
    'axes.labelsize': 11, 'axes.titlesize': 12,
})

C_RF, C_LO, C_I, C_Q, C_BB, C_BIN = (
    'steelblue', '#9467bd', '#d62728', '#2ca02c', '#ff7f0e', '#7f7f7f')

fs = 4000.0                       # ADC sampling rate (Hz)
T  = 0.25                         # record / pulse duration (s)
t  = np.arange(0, T, 1 / fs)      # time axis
N  = len(t)                       # samples -> FFT length

def make_pulse(fc, width_frac=0.12, phi=0.0):
    # Real bandpass Gaussian-windowed tone at carrier fc
    env = np.exp(-((t - T / 2) ** 2) / (2 * (width_frac * T) ** 2))
    return env * np.cos(2 * np.pi * fc * t + phi), env

def spectrum(x):
    # Two-sided magnitude spectrum, centred (DC in the middle)
    X = np.fft.fftshift(np.fft.fft(x)) / N
    f = np.fft.fftshift(np.fft.fftfreq(N, 1 / fs))
    return f, np.abs(X)

print(f'fs = {fs:.0f} Hz   N = {N} samples   FFT bin width = fs/N = {fs/N:.2f} Hz')

fs = 4000 Hz   N = 1000 samples   FFT bin width = fs/N = 4.00 Hz


## The Received Pulse and Its Carrier

The Rx sees a real, band-limited pulse: a slow **envelope** $e(t)$ riding on a fast **carrier** $\cos(2\pi f_c t)$. In the spectrum this places two mirror-image lobes at $\pm f_c$, each shaped by the envelope's transform. Sliding $f_c$ slides both lobes along the frequency axis.

$$x(t) = e(t)\,\cos(2\pi f_c t)\quad\Longleftrightarrow\quad X(f) = \tfrac12\big[E(f-f_c) + E(f+f_c)\big]$$

In [2]:
def show_carrier(fc, width):
    x, env = make_pulse(fc, width)
    f, S = spectrum(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))

    ax1.plot(t * 1e3, x, color=C_RF, lw=0.9)
    ax1.plot(t * 1e3,  env, color='k', lw=1.4, ls='--', alpha=0.7)
    ax1.plot(t * 1e3, -env, color='k', lw=1.4, ls='--', alpha=0.7)
    ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('Amplitude')
    ax1.set_title(f'$x(t)$ — carrier {fc:.0f} Hz under Gaussian envelope')

    ax2.plot(f, S, color=C_RF, lw=1.6)
    ax2.fill_between(f, S, alpha=0.2, color=C_RF)
    for s in (+1, -1):
        ax2.axvline(s * fc, color=C_RF, ls=':', lw=1.2, alpha=0.7)
    ax2.set_xlim(-fs / 2, fs / 2)
    ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|X(f)|')
    ax2.set_title('Mirror lobes at $\\pm f_c$')
    plt.tight_layout(); plt.show()

fc_w    = widgets.FloatSlider(value=600, min=100, max=1800, step=20,
            description='f_c  carrier (Hz)',
            style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
width_w = widgets.FloatSlider(value=0.12, min=0.04, max=0.25, step=0.01,
            description='envelope width',
            style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
widgets.interactive_output(show_carrier, {'fc': fc_w, 'width': width_w})
display(widgets.VBox([widgets.HBox([fc_w, width_w]),
        widgets.interactive_output(show_carrier, {'fc': fc_w, 'width': width_w})]))

## Quadrature Mixing: RF $\to$ IF $\to$ Baseband

Multiplying the pulse by a local oscillator at $f_{LO}$ shifts every spectral line by $\pm f_{LO}$, producing a **difference** term (down to IF or DC) and a **sum** image at $f_c + f_{LO}$. The image is removed by the low-pass filter, leaving the wanted band centred at the intermediate frequency $f_{IF}=f_c-f_{LO}$:

$$x(t)\cos(2\pi f_{LO}t) = \tfrac12\,e(t)\big[\cos(2\pi (f_c-f_{LO})t) + \cos(2\pi (f_c+f_{LO})t)\big]$$

Set $f_{LO}=f_c$ for true zero-IF (homodyne); set $f_{LO}<f_c$ to park the pulse at a chosen IF.

In [ ]:
def lowpass(x, fcut):
    X = np.fft.rfft(x)
    fr = np.fft.rfftfreq(len(x), 1 / fs)
    X[fr > fcut] = 0.0
    return np.fft.irfft(X, n=len(x))

def show_mixer(fc, fLO, fcut):
    x, _ = make_pulse(fc)
    mix  = x * np.cos(2 * np.pi * fLO * t)        # real mixer output
    flt  = lowpass(mix, fcut)
    fIF  = abs(fc - fLO)

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
    for ax, sig, ttl, col in [
        (axes[0], x,   f'RF in @ {fc:.0f} Hz',                      C_RF),
        (axes[1], mix, f'After mixer: IF {fIF:.0f} + image {fc+fLO:.0f}', C_LO),
        (axes[2], flt, f'After LPF ({fcut:.0f} Hz): IF kept',        C_BB)]:
        f, S = spectrum(sig)
        ax.plot(f, S, color=col, lw=1.5)
        ax.fill_between(f, S, alpha=0.2, color=col)
        ax.set_xlim(-fs / 2, fs / 2); ax.set_xlabel('Frequency (Hz)')
        ax.set_title(ttl, fontsize=10)
    axes[1].axvline(fc + fLO, color='k', ls=':', lw=1.1, alpha=0.6)
    axes[2].axvline(fcut, color='k', ls='--', lw=1, alpha=0.5)
    axes[2].axvline(-fcut, color='k', ls='--', lw=1, alpha=0.5)
    axes[0].set_ylabel('|X(f)|'); plt.tight_layout(); plt.show()

s = dict(style={'description_width': '140px'}, layout=widgets.Layout(width='430px'))
fc2  = widgets.FloatSlider(value=600, min=200, max=1500, step=20, description='f_c  carrier (Hz)', **s)
fLO2 = widgets.FloatSlider(value=600, min=200, max=1500, step=20, description='f_LO  mixer (Hz)', **s)
cut2 = widgets.FloatSlider(value=150, min=40, max=400, step=10, description='LPF cutoff (Hz)', **s)
display(widgets.VBox([widgets.HBox([fc2, fLO2, cut2]),
        widgets.interactive_output(show_mixer, {'fc': fc2, 'fLO': fLO2, 'fcut': cut2})]))

## The Residual $\Delta f$ Problem — One Cell, Every Stage

When $f_{LO}\neq f_c$ a residual $\Delta f = f_c - f_{LO}$ leaks through. The complex baseband becomes a pure rotating phasor whose I and Q are cosines/sines at $\Delta f$ — the larger $|\Delta f|$, the more oscillations crammed under the envelope:

$$z(t) = I(t)+jQ(t) = \tfrac12\,e(t)\,e^{\,j(2\pi \Delta f\, t + \phi)}\;\Rightarrow\; I=\tfrac12 e\cos(2\pi\Delta f t+\phi),\;\; Q=\tfrac12 e\sin(2\pi\Delta f t+\phi)$$

This single interactive panel shows all four views at once and updates them **together** when any slider moves: (1) RF spectrum with the $\pm f_c$ lobes and the $f_{LO}$ marker, (2) the demodulated I/Q in time, (3) the baseband spectrum with the residual peak walking off DC, and (4) the **DFT bin grid** so you can see exactly which bin the pulse falls into.

In [4]:
def iq_chain(fc, fLO, phi):
    # Full Rx chain: returns time I,Q and complex baseband z
    x, env = make_pulse(fc, phi=phi)
    I = lowpass(x * np.cos(2 * np.pi * fLO * t),  120)
    Q = lowpass(x * (-np.sin(2 * np.pi * fLO * t)), 120)
    return x, env, I, Q, (I + 1j * Q)

def master(fc, fLO, phi, bin_zoom):
    df = fc - fLO
    x, env, I, Q, z = iq_chain(fc, fLO, phi)

    f_rf, S_rf = spectrum(x)
    f_bb, S_bb = spectrum(z)
    bin_w = fs / N                         # DFT bin width
    k_peak = int(round(df / bin_w))        # bin index the residual lands in
    on_grid = abs(df / bin_w - round(df / bin_w)) < 1e-9

    fig = plt.figure(figsize=(14, 7.4))
    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.24)
    axRF, axIQ = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    axBB, axBN = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

    # (1) RF spectrum with carrier lobes + LO line
    axRF.plot(f_rf, S_rf, color=C_RF, lw=1.5)
    axRF.fill_between(f_rf, S_rf, alpha=0.2, color=C_RF)
    for sgn in (+1, -1):
        axRF.axvline(sgn * fc, color=C_RF, ls=':', lw=1.1, alpha=0.7)
    axRF.axvline(fLO, color=C_LO, ls='--', lw=1.6, label=f'$f_{{LO}}$={fLO:.0f}')
    axRF.set_xlim(-fs / 2, fs / 2)
    axRF.set_xlabel('Frequency (Hz)'); axRF.set_ylabel('|X(f)|')
    axRF.set_title(f'RF spectrum — lobes at $\\pm${fc:.0f} Hz')
    axRF.legend(fontsize=9, loc='upper right')

    # (2) demodulated I/Q in time — oscillations ~ df*T
    axIQ.plot(t * 1e3, I, color=C_I, lw=1.3, label='I(t)')
    axIQ.plot(t * 1e3, Q, color=C_Q, lw=1.3, label='Q(t)')
    axIQ.plot(t * 1e3,  env / 2, color='k', ls='--', lw=1, alpha=0.5)
    axIQ.plot(t * 1e3, -env / 2, color='k', ls='--', lw=1, alpha=0.5)
    axIQ.set_xlabel('Time (ms)'); axIQ.set_ylabel('Amplitude')
    axIQ.set_title(f'Baseband I/Q — $\\Delta f$={df:+.0f} Hz '
                   f'$\\Rightarrow$ {abs(df)*T:.1f} cycles')
    axIQ.legend(fontsize=9, loc='upper right')

    # (3) baseband spectrum — residual peak walks off DC
    axBB.plot(f_bb, S_bb, color=C_BB, lw=1.6)
    axBB.fill_between(f_bb, S_bb, alpha=0.25, color=C_BB)
    axBB.axvline(0, color='k', lw=0.7)
    axBB.axvline(df, color=C_I, ls='--', lw=1.6, label=f'$\\Delta f$={df:+.0f} Hz')
    axBB.set_xlim(-150, 150)
    axBB.set_xlabel('Frequency (Hz)'); axBB.set_ylabel('|Z(f)|')
    axBB.set_title('Complex-baseband spectrum (residual off DC)')
    axBB.legend(fontsize=9, loc='upper right')

    # (4) DFT bin grid — where does the pulse fall?
    span = bin_zoom * bin_w
    centres = np.arange(-round(span / bin_w), round(span / bin_w) + 1) * bin_w
    mask = (f_bb >= -span) & (f_bb <= span)
    axBN.plot(f_bb[mask], S_bb[mask], color=C_BB, lw=1.4, zorder=3)
    for c in centres:
        axBN.axvline(c, color=C_BIN, lw=0.8, alpha=0.45)
        axBN.axvspan(c - bin_w / 2, c + bin_w / 2,
                     color=C_BIN, alpha=0.05, zorder=0)
    axBN.axvline(k_peak * bin_w, color=C_I, lw=2.4, alpha=0.5,
                 label=f'bin k={k_peak:+d}')
    axBN.axvline(df, color='k', ls='--', lw=1.2,
                 label=('on-grid' if on_grid else 'between bins'))
    axBN.set_xlim(-span, span)
    axBN.set_xlabel('Frequency (Hz)'); axBN.set_ylabel('|Z(f)|')
    axBN.set_title(f'DFT bins (width {bin_w:.1f} Hz) — '
                   + ('energy in ONE bin' if on_grid else 'leaks across bins'))
    axBN.legend(fontsize=9, loc='upper right')
    plt.show()

m = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='470px'))
fcM  = widgets.FloatSlider(value=600, min=200, max=1500, step=10,  description='f_c  carrier (Hz)', **m)
fLOM = widgets.FloatSlider(value=600, min=200, max=1500, step=5,   description='f_LO  mixer (Hz)', **m)
phiM = widgets.FloatSlider(value=0.0, min=0, max=2*np.pi, step=0.1, description='φ  phase (rad)', **m)
zoomM= widgets.IntSlider(value=6, min=2, max=14, step=1,           description='bin-grid zoom (±k)', **m)
display(widgets.VBox([widgets.HBox([fcM, fLOM]), widgets.HBox([phiM, zoomM]),
        widgets.interactive_output(master,
            {'fc': fcM, 'fLO': fLOM, 'phi': phiM, 'bin_zoom': zoomM})]))

## When Does a Pulse Land in One Bin vs. Leak?

A residual that is an **exact integer multiple** of the bin width $f_s/N$ lands fully inside a single DFT bin; otherwise its energy **scallops** across neighbours (spectral leakage). Tune $\Delta f$ continuously and watch a single tone slide off its bin centre.

| Residual $\Delta f$ | I/Q time view | Bin grid |
|:---|:---|:---|
| $0$ | flat (DC) | all energy in bin $0$ |
| integer $\times f_s/N$ | clean cosines | one bin, no leakage |
| half-bin offset | cosines | maximum leakage to two bins |

In [6]:
def bin_focus(df, win):
    # Single complex tone at df with optional Hann window
    env = np.hanning(N) if win else np.ones(N)
    z = env * np.exp(1j * 2 * np.pi * df * t)
    f, S = spectrum(z)
    bin_w = fs / N
    frac = df / bin_w
    on_grid = abs(frac - round(frac)) < 1e-9

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.9))
    a1.plot(t * 1e3, z.real, color=C_I, lw=1.2, label='I')
    a1.plot(t * 1e3, z.imag, color=C_Q, lw=1.2, label='Q')
    a1.set_xlabel('Time (ms)'); a1.set_ylabel('Amplitude')
    a1.set_title(f'Tone at $\\Delta f$={df:.1f} Hz = {frac:.2f}·bins')
    a1.legend(fontsize=9, loc='upper right')

    span = 7 * bin_w
    mask = (f >= df - span) & (f <= df + span)
    mk, st, _ = a2.stem(f[mask] / bin_w, S[mask],
                        linefmt='-', markerfmt='o', basefmt=' ')
    plt.setp(st, color=C_BB, lw=1.2)          # hex color set here, not in linefmt
    plt.setp(mk, color=C_BB, ms=5)
    a2.axvline(frac, color='k', ls='--', lw=1.3,
               label=('on a bin' if on_grid else 'between bins'))
    a2.set_xlabel('Bin index  (f / bin width)'); a2.set_ylabel('|Z|')
    a2.set_title('Leakage ' + ('NONE' if on_grid else 'across bins'))
    a2.legend(fontsize=9, loc='upper right')
    plt.tight_layout(); plt.show()

bw = fs / N
b = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='470px'))
dfB  = widgets.FloatSlider(value=2 * bw, min=0, max=10 * bw, step=bw / 20,
        description=f'Δf (bin={bw:.1f} Hz)', readout_format='.1f', **b)
winB = widgets.Checkbox(value=False, description='apply Hann window',
        style={'description_width': '150px'})
display(widgets.VBox([widgets.HBox([dfB, winB]),
        widgets.interactive_output(bin_focus, {'df': dfB, 'win': winB})]))